# **Don Quixote de la Mancha**

Ce notebook présente une exploration de la librairie SpaCy pour l'ouvrage de Miguel de Cervantes : *Don Quixote de la Mancha*.

Pour commencer, il faut charger la bibliothèque puis le modèle de langue en espagnol de taille moyenne :

In [2]:
import spacy


Après avoir téléchargé le modèle il faut le charger :

In [3]:
nlp_es = spacy.load("es_core_news_md")

Il faut télécharger le texte afin de voir les opérations à effectuer :

In [4]:
import requests
url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"

def get_book(url):
    don_quixote = "don_quixote.txt"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(don_quixote, "w",encoding="utf-8") as file:
            file.write(response.text)
        print("Le livre 'Don Quixote' a bien été récupéré.")
    except requests.exceptions.RequestException as exception:
        print(f"Le livre n'a pas été téléchargé car {exception}")

get_book("https://www.gutenberg.org/cache/epub/2000/pg2000.txt")

Le livre 'Don Quixote' a bien été récupéré.


Il faut enlever tout ce qui ne fait pas partie du roman :

In [5]:
import re

# Chargement du fichier brut
with open("don_quixote.txt", "r", encoding="utf-8") as f:
    raw_content = f.read()

# Expression régulière pour isoler le corps du texte
# Utilisation de group(1) pour récupérer ce qui est entre les parenthèses capturantes
pattern = r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*(.*?)\*\*\* END OF THE PROJECT GUTENBERG EBOOK"

match = re.search(pattern, raw_content, re.DOTALL | re.IGNORECASE)

if match:
    cleaned_text = match.group(1).strip()
else:
    # Fallback si les balises sont absentes ou mal formatées
    cleaned_text = raw_content.strip()

# Affichage des métriques de nettoyage
print(f"Caractères avant : {len(raw_content)}")
print(f"Caractères après : {len(cleaned_text)}")

# Sauvegarde locale pour éviter de recharger le fichier brut à chaque fois
with open("don_quixote_clean.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)

Caractères avant : 2130058
Caractères après : 2110726


Contrairement à **l'Art de la Guerre** où l'introduction contient des éléments historiques modernes qui ne sont pas écrits par **Sun Tzu**, **Don Quixote** a écrit les poèmes et prologues sont écrits par **Miguel de Cervantes**. Les supprimer de notre corps de texte signifierait donc à supprimer une partie de son oeuvre. **Miguel de Cervantes** a écrit ces poèmes, faux témoignages, lettres et poèmes parodiques (en faisant croire qu'ils étaient écrits par des personnages de légendes) pour se moquer des auteurs qui en son temps cherchaient à obtenir des parrainages prestigieux. Il a notamment écrit un poème en se faisant passer pour Amadis de Gaula, le héros de roman de chevalerie espagnol le plus célèbre.

De plus, conserver l'intégralité du corps du texte permettra de tester la robustesse du modèle en observant sa gestion du passage de la poésie à la prose narrative ainsi que l'alternance entre le ton solonel. L'approche pour **Don Quixote** est holistique i.e. l'objectif est de capturer la richesse de la langue espagnol au 17ème siècle. Cela permettra en même temps si spaCy est perfomant sur les textes anciens que sur le texte qui est rédigé dans un style plus moderne.

## **2. La Tokenisation**

Maintenant que le texte est près, il faut lancer toute la pipeline spaCy sur le texte. Ce qu'il fallait faire en plusieurs étapes avec NLTK se fait en une seule ligne de commande :

In [6]:
quixote_doc = nlp_es(cleaned_text)

ValueError: [E088] Text of length 2110726 exceeds maximum of 1000000. The parser and NER models require roughly 1GB of temporary memory per 100,000 characters in the input. This means long texts may cause memory allocation errors. If you're not using the parser or NER, it's probably safe to increase the `nlp.max_length` limit. The limit is in number of characters, so you can check whether your inputs are too long by checking `len(text)`.

Le texte a une longueur de 2 130 058 caractères, la limite standard de spaCy pour la pipeline est de 1 000 000 de caractères, cela rend pour l'instant impossible l'application de la pipeline au texte. Il faudra réessayer en modifiant la limite standard

In [7]:
# 1. Modification de la limite de caractères du modèle.
nlp_es.max_length = 2500000 

# 2. Découpage par chapitres
# On retire (?i) du pattern et on utilise flags=re.IGNORECASE
pattern_chapitre = r"(CAPÍTULO\s+[IVXLCDM\d]+)"

# Utilisation du paramètre flags pour la gestion majuscules/minuscules
segments_bruts = re.split(pattern_chapitre, cleaned_text, flags=re.IGNORECASE)

# Nettoyage des segments et suppression des entrées vides
sections_quijote = [s.strip() for s in segments_bruts if s.strip()]

# 3. Traitement via nlp.pipe
quijote_docs = list(nlp_es.pipe(sections_quijote, n_process=-1, batch_size=20))

print(f"Nombre de sections traitées : {len(quijote_docs)}")

Nombre de sections traitées : 259


Après avoir utilisé nlp_es.pipe, la pipeline a réussi à traiter 259 sections. Il faut maintenant voir si la structure de notre documents est cohérente :

In [8]:
for i, doc in enumerate(quijote_docs[:10]):
    print(f"Index {i} Longueur : {len(doc)} tokens - Début : {doc.text[:50]}")

Index 0 Longueur : 10976 tokens - Début : El ingenioso hidalgo don Quijote de la Mancha



p
Index 1 Longueur : 2 tokens - Début : Capítulo II
Index 2 Longueur : 2762 tokens - Début : . Que trata de la primera salida que de su tierra 
Index 3 Longueur : 2 tokens - Début : Capítulo III
Index 4 Longueur : 2868 tokens - Début : . Donde se cuenta la graciosa manera que tuvo don 
Index 5 Longueur : 2 tokens - Début : Capítulo IV
Index 6 Longueur : 3106 tokens - Début : . De lo que le sucedió a nuestro caballero cuando 
Index 7 Longueur : 2 tokens - Début : Capítulo V
Index 8 Longueur : 2015 tokens - Début : . Donde se prosigue la narración de la desgracia d
Index 9 Longueur : 2 tokens - Début : Capítulo VI


In [9]:
# Vérification des 6 derniers segments (environ 3 chapitres complets)
print(f"Total de segments dans quijote_docs : {len(quijote_docs)}\n")

for i in range(-6, 0):
    section = quijote_docs[i]
    # On affiche l'index réel, le nombre de tokens et le début du texte
    # L'index réel est len(quijote_docs) + i
    index_reel = len(quijote_docs) + i
    
    print(f"Index : {index_reel}")
    print(f"Nombre de tokens : {len(section)}")
    print(f"Extrait : {section.text[:100]}")
    print("-" * 30)

Total de segments dans quijote_docs : 259

Index : 253
Nombre de tokens : 2
Extrait : Capítulo LXXII
------------------------------
Index : 254
Nombre de tokens : 2247
Extrait : . De cómo don Quijote y Sancho llegaron a su aldea

Todo aquel día, esperando la noche, estuvieron e
------------------------------
Index : 255
Nombre de tokens : 2
Extrait : Capítulo LXXIII
------------------------------
Index : 256
Nombre de tokens : 2203
Extrait : . De los agüeros que tuvo don Quijote al entrar de su aldea,
con otros sucesos que adornan y acredit
------------------------------
Index : 257
Nombre de tokens : 2
Extrait : Capítulo LXXIV
------------------------------
Index : 258
Nombre de tokens : 3076
Extrait : . De cómo don Quijote cayó malo, y del testamento que hizo, y
su muerte

Como las cosas humanas no s
------------------------------


## **III. Exploration du texte**

Maintenant que notre texte est bien segmenté et traité par spaCy, l'exploitation des outils du modèle peut commencer. La première étape sera de voir si le modèle est capable de reconnaître les personnages de l'oeuvre. Pour ce faire mais également afin de voir s'il y a des personnages qui apparaissent dans le corps du texte et dans les textes satiriques :

In [10]:
'''quijote_doc est le texte final tokenizé. C'est une liste 
de listes dont les éléments avec un index pair contiennent 
du texte et les éléments avec  un index impair 
contiennent les noms de chapitres'''

from collections import Counter

partie_satirique = quijote_docs[0]

# On extrait la liste des personnages qui sont présents dans les textes parodiques :

personnages_intro = [element.text for element in partie_satirique.ents if element.label_ == 'PER']

# On crée une variable freq_count qui comptera les personnages présents :
freq_front = Counter(personnages_intro)

# On affiche les dix personnages les plus présents :

print("TOP 10 des personnages les plus fréquents")
for nom,count in freq_front.most_common(10):
    # pour des raisons de clareté visuelle, on met 25 en paramètres de largeur de colonne fixes pour être large
    # le paramètre ^ sert à forcer l'alignement au centre
    print(f"{nom:^25} apparaît {count} fois.")


TOP 10 des personnages les plus fréquents
      Sancho Panza        apparaît 18 fois.
        Dulcinea          apparaît 6 fois.
         Sancho           apparaît 5 fois.
       Aristóteles        apparaît 4 fois.
Miguel de Cervantes Saavedra apparaît 3 fois.
  Juan Gallo de Andrada   apparaît 3 fois.
          Dios            apparaît 3 fois.
         Amadís           apparaît 3 fois.
     Amadís de Gaula      apparaît 3 fois.
         Prólogo          apparaît 2 fois.


In [11]:
len(freq_front)

116

SpaCy a détecté 116 personnages dans la partie parodiques mais on observe dans le top 10 des personnages que des personnages apparaissent deux fois car ils ne sont pas mentionnés de la même manières (Sancho, Sancho Panza) ou (Amadis de Gaula, Amadis)

In [12]:


# 2. Heuristique simple : Regrouper si un nom est inclus dans un autre
# On trie par longueur décroissante pour comparer les noms longs aux noms courts
noms_uniques_tries = sorted(list(set(freq_front)), key=len, reverse=True)
mapping_automatique = {}

for i, nom_long in enumerate(noms_uniques_tries):
    for nom_court in noms_uniques_tries[i+1:]:
        if nom_court in nom_long:
            mapping_automatique[nom_court] = nom_long

# 3. Application du mapping et comptage final
noms_nettoyes = [mapping_automatique.get(n, n) for n in freq_front]
frequence_corrigee = Counter(noms_nettoyes)

# 4. Affichage des résultats après réduction
print(f"Nombre d'entités après regroupement par inclusion : {len(frequence_corrigee)}")
print("\nTop 15 des personnages normalisés :")
for nom, count in frequence_corrigee.most_common(15):
    print(f"{nom:30} | {count}")

Nombre d'entités après regroupement par inclusion : 106

Top 15 des personnages normalisés :
Miguel de Cervantes Saavedra   | 2
Sancho Panza

Del progreso     | 2
Sancho
Panza                   | 2
Dulcinea del Tobo-             | 2
San Basilio                    | 2
Altisidora

Donde              | 2
doña Rodríguez

Del            | 2
Francisco Murcia de la Llana   | 2
Ovidio os                      | 2
Amadís de Gaula                | 2
Al Duque de Béjar              | 1
Prólogo                        | 1
Marcela                        | 1
Mambrino                       | 1
Beltenebros                    | 1


In [13]:
import re
from collections import Counter

# 1. Extraction brute des entités PER de l'introduction
doc_intro = quijote_docs[0]
entites_brutes = [ent.text.replace('\n', ' ').strip() for ent in doc_intro.ents if ent.label_ == "PER"]

# 2. Nettoyage initial : ponctuation et bruits fréquents
# On retire la ponctuation collée aux noms et les termes qui ne sont pas des personnages
bruit_ner = {"Del", "Donde", "Que", "Al", "Prólogo", "Capítulo", "Folio"}

entites_nettoyees = []
for nom in entites_brutes:
    # Nettoyage de la ponctuation finale (points, virgules, points d'interrogation)
    nom_propre = re.sub(r'[?.,;!:]+$', '', nom)
    if nom_propre not in bruit_ner and len(nom_propre) > 2:
        entites_nettoyees.append(nom_propre)

# 3. Réduction des doublons par inclusion
# On identifie les formes uniques et on les trie par longueur (décroissant)
noms_uniques_tries = sorted(list(set(entites_nettoyees)), key=len, reverse=True)
mapping_normalisation = {}

for i, nom_long in enumerate(noms_uniques_tries):
    for nom_court in noms_uniques_tries[i+1:]:
        # Si le nom court est contenu dans le nom long, on crée un lien
        if nom_court in nom_long:
            mapping_normalisation[nom_court] = nom_long

# 4. Application du mapping et comptage
intro_normalisee = [mapping_normalisation.get(n, n) for n in entites_nettoyees]
stats_intro = Counter(intro_normalisee)

# 5. Affichage du résultat
print(f"Nombre d'entités uniques après réduction : {len(stats_intro)}")
print("-" * 50)
print(f"{'Personnage':<35} | {'Occurrences'}")
print("-" * 50)
for personnage, freq in stats_intro.most_common(15):
    print(f"{personnage:<35} | {freq}")

Nombre d'entités uniques après réduction : 102
--------------------------------------------------
Personnage                          | Occurrences
--------------------------------------------------
Sancho Panza  Del progreso          | 21
Dulcinea del Tobo-                  | 7
Amadís de Gaula                     | 6
Sancho Panza                        | 5
Miguel de Cervantes Saavedra        | 4
Aristóteles                         | 4
doña Rodríguez  Del                 | 3
Juan Gallo de Andrada               | 3
Dios                                | 3
Mambrino                            | 2
Sansón Carrasco                     | 2
Teresa Panza                        | 2
Dulcinea del Toboso                 | 2
Camacho                             | 2
San Basilio                         | 2


In [14]:
entites_nettoyees

['Miguel de Cervantes Saavedra',
 'Al Duque de Béjar',
 'Marcela',
 'Sancho Panza',
 'Sancho Panza',
 'Sancho',
 'Mambrino',
 'Beltenebros',
 'Dorotea',
 'Sancho Panza',
 'Mambrino',
 'Sancho Panza',
 'Fee',
 'conde de Lemos',
 'Sancho Panza',
 'Sancho Panza',
 'Sansón Carrasco',
 'Sancho Panza',
 'Sansón Carrasco',
 'Sancho Panza',
 'Teresa Panza',
 'Dulcinea del Toboso',
 'Sancho',
 'Dulcinea',
 'Camacho',
 'Basilio',
 'Camacho',
 'Montesinos',
 'Montesinos',
 'Pedro',
 'Benengeli',
 'Sancho Panza',
 'Dulcinea del Toboso',
 'Dulcinea',
 'Sancho Panza',
 'Teresa Panza',
 'Sancho Panza',
 'Sancho Panza  Cómo Sancho Panza',
 'Sancho Panza',
 'Altisidora  Donde',
 'Sancho Panza',
 'doña Rodríguez',
 'Sancho Panza',
 'Teresa Sancha',
 'Sancho Panza  Del progreso',
 'Sancho Panza',
 'Angustiada',
 'doña Rodríguez  Del',
 'Sancho Panza',
 'Sancho',
 'doña Rodríguez',
 'Altisidora',
 'Sancho Panza',
 'Don Gregorio',
 'Sancho',
 'Sancho',
 'Juan Gallo de Andrada',
 'Miguel de Cervantes Saaved

Malgré deux tentatives pour nettoyer la liste des personnages, le constat est sans appel : SpaCy qui est entrainé sur de l'espagnol moderne, confond beaucoup de verbes avec des noms propres ("Decia","Envidio","Améla","Fama"). Pour nettoyer au maximum la liste de personnages, il faudra ajouter un filtre de nature grammaticale (POS tagging). L'entité détectée ne sera gardée dans la liste de personnages seulement si spaCy ne la détecte pas comme étant un verbe :

In [ ]:
import re
from collections import Counter

# À exécuter avant la cellule de filtrage
alias_map = {
    "Panza": "Sancho Panza",
    "Sancho": "Sancho Panza",
    "Quijote": "Don Quijote",
    "Quijada": "Don Quijote",
    "Quesada": "Don Quijote",
    "Dulcinea del Tobo-": "Dulcinea del Toboso",
    "Dulcineas": "Dulcinea del Toboso"
}

# Liste de "faux personnages" (verbes ou bruits récurrents)
faux_personnages = {
    "Decía", "Dijo", "Respondió", "Parece", "Llamábase", "Améla", 
    "Llenósele", "Rey", "Fama", "Febo", "Apolo", "Fortuna"
}

def filtrage_avance(entite):
    texte = entite.text.replace('\n', ' ').strip()
    
    # 1. On ignore si l'entité est dans notre liste noire
    if texte in faux_personnages:
        return None
        
    # 2. On ignore si spaCy a tagué le mot principal comme un VERBE
    # (Utile pour les mots comme 'engendró', 'quisiéredes', etc.)
    if len(entite) == 1 and entite[0].pos_ == "VERB":
        return None

    # 3. Nettoyage des scories (tirets de césure et suffixes)
    texte = re.sub(r'[-\s]+$', '', texte)
    texte = re.sub(r'\s+(Del|Donde|os)$', '', texte, flags=re.IGNORECASE)
    
    return texte if len(texte) > 2 else None

# Traitement de l'introduction
personnages_intro_propres = []

for ent in quijote_docs[0].ents:
    if ent.label_ == "PER":
        nom_nettoye = filtrage_avance(ent)
        if nom_nettoye:
            # Application de tes alias (Sancho, Quijote, etc.)
            nom_final = alias_map.get(nom_nettoye, nom_nettoye)
            personnages_intro_propres.append(nom_final)

stats_intro = Counter(personnages_intro_propres)

# Affichage du Top 15
print(f"{'Personnage':<35} | {'Occurrences'}")
print("-" * 50)
for personnage, freq in stats_intro.most_common(15):
    print(f"{personnage:<35} | {freq}")

Personnage                          | Occurrences
--------------------------------------------------
Sancho Panza                        | 25
Dulcinea                            | 6
Aristóteles                         | 4
Miguel de Cervantes Saavedra        | 3
doña Rodríguez                      | 3
Juan Gallo de Andrada               | 3
Dios                                | 3
Amadís                              | 3
Amadís de Gaula                     | 3
Prólogo                             | 2
Mambrino                            | 2
Sansón Carrasco                     | 2
Teresa Panza                        | 2
Dulcinea del Toboso                 | 2
Camacho                             | 2


In [16]:
print(len(stats_intro), stats_intro, sep= '/n')

95/nCounter({'Sancho Panza': 25, 'Dulcinea': 6, 'Aristóteles': 4, 'Miguel de Cervantes Saavedra': 3, 'doña Rodríguez': 3, 'Juan Gallo de Andrada': 3, 'Dios': 3, 'Amadís': 3, 'Amadís de Gaula': 3, 'Prólogo': 2, 'Mambrino': 2, 'Sansón Carrasco': 2, 'Teresa Panza': 2, 'Dulcinea del Toboso': 2, 'Camacho': 2, 'Montesinos': 2, 'Altisidora': 2, 'Ovidio': 2, 'Alejandro': 2, 'Orlando': 2, 'Don Quijote': 2, 'Al Duque de Béjar': 1, 'Marcela': 1, 'Beltenebros': 1, 'Dorotea': 1, 'Fee': 1, 'conde de Lemos': 1, 'Basilio': 1, 'Pedro': 1, 'Benengeli': 1, 'Sancho Panza  Cómo Sancho Panza': 1, 'Teresa Sancha': 1, 'Sancho Panza  Del progreso': 1, 'Angustiada': 1, 'Don Gregorio': 1, 'Francisco Murcia de la Llana': 1, 'Miguel de Cervantes': 1, 'sentenciare': 1, 'Juan de Amezqueta': 1, 'PRÓLOGO  Desocupado': 1, 'Platón': 1, 'Zeuxis': 1, 'quisiéredes': 1, 'Preste Juan de las Indias': 1, 'sacáredes': 1, 'pusiéredes': 1, 'Horacio': 1, 'Catón': 1, 'Goliat': 1, 'David': 1, 'Lamia': 1, 'Laida': 1, 'Medea': 1, 'Hom

La liste de personnages a été réduites mais certains verbes issus du futur du subjonctif comme "alcanzáredes", "quisiéredes" ou "Aristóteles" ne sont toujours pas détectés par spaCy. SpaCy ne voit comprend pas que Dulcinea del Tobo et Duclinea del Toboso est identique. On a aussi "Sancho Panza Cómo Sancho Panza" qui est compté comme un personnage à part entière. Après avoir essayé de nettoyer au maximum la liste des personnages, il reste un dernier recours : Comparer notre listes de personnages avec les personnages listés sur Wikipédia. Pour ce faire, il faudra interroger l'API de Wikipédia : Wikidata

In [ ]:
import requests
import time

def valider_entite_wikidata(nom_entite):
    url = "https://www.wikidata.org/w/api.php"
    
    # INDISPENSABLE : Identifier la requête pour ne pas être bloqué
    headers = {
        'User-Agent': 'ProjetEtudeMasterLorame/1.0 (contact: lorame@exemple.com)'
    }
    
    params = {
        "action": "wbsearchentities",
        "language": "es",
        "format": "json",
        "search": nom_entite,
        "type": "item"
    }
    
    try:
        reponse = requests.get(url, params=params, headers=headers, timeout=5)
        if reponse.status_code == 200:
            donnees = reponse.json()
            if donnees.get("search"):
                # On prend le premier résultat (le plus pertinent)
                match = donnees["search"][0]
                return {
                    "id": match["id"],
                    "label": match["label"],
                    "description": match.get("description", "No hay descripción disponible")
                }
        else:
            print(f"Erreur HTTP {reponse.status_code} pour : {nom_entite}")
    except Exception as e:
        print(f"Erreur de connexion pour {nom_entite} : {e}")
    
    return None

# Dictionnaire pour stocker les validations
resultats_validation = {}

print(f"Début de la validation sur Wikidata pour {len(stats_intro)} noms...")

# On limite pour le test aux 20 premiers pour ne pas saturer l'API immédiatement
for i, nom in enumerate(stats_intro.keys()):
    # Petite pause pour respecter l'API (Rate Limiting)
    time.sleep(0.1) 
    
    info_wiki = valider_entite_wikidata(nom)
    
    if info_wiki:
        resultats_validation[nom] = info_wiki
        print(f" OK Trouvé : {nom} -> {info_wiki['label']} ({info_wiki['id']})")
    else:
        print(f"KO - Non trouvé : {nom}")

print(f"\nValidation terminée. {len(resultats_validation)} entités validées sur {len(stats_intro)}.")

Début de la validation sur Wikidata pour 95 noms...
 OK Trouvé : Miguel de Cervantes Saavedra -> Miguel de Cervantes (Q5682)
KO - Non trouvé : Al Duque de Béjar
 OK Trouvé : Prólogo -> prologue (Q485321)
 OK Trouvé : Marcela -> Marcela (Q19798827)
 OK Trouvé : Sancho Panza -> Sancho Panza (Q630823)
 OK Trouvé : Mambrino -> Mambrino (Q6745553)
 OK Trouvé : Beltenebros -> Prince of Shadows (Q4366360)
 OK Trouvé : Dorotea -> Dorotea Municipality (Q132334)
 OK Trouvé : Fee -> Antoine Laurent Apollinaire Fée (Q2627241)
 OK Trouvé : conde de Lemos -> Count of Lemos (Q8349573)
 OK Trouvé : Sansón Carrasco -> Sansón Carrasco (Q6120178)
 OK Trouvé : Teresa Panza -> Teresa Panza (Q56634053)
 OK Trouvé : Dulcinea del Toboso -> Dulcinea (Q1264776)
 OK Trouvé : Dulcinea -> Dulcinea (Q911015)
 OK Trouvé : Camacho -> Camacho (Q21482925)
 OK Trouvé : Basilio -> Basilio (Q18416923)
 OK Trouvé : Montesinos -> Montesinos (Q118810278)
 OK Trouvé : Pedro -> Pedro (Q15897419)
 OK Trouvé : Benengeli -> Cide 

In [18]:
resultats_validation

{'Miguel de Cervantes Saavedra': {'id': 'Q5682',
  'label': 'Miguel de Cervantes',
  'description': 'Spanish novelist, poet, and playwright (1547-1616)'},
 'Prólogo': {'id': 'Q485321', 'label': 'prologue', 'description': 'cycling'},
 'Marcela': {'id': 'Q19798827',
  'label': 'Marcela',
  'description': 'female given name'},
 'Sancho Panza': {'id': 'Q630823',
  'label': 'Sancho Panza',
  'description': 'fictional character in the novel Don Quixote'},
 'Mambrino': {'id': 'Q6745553',
  'label': 'Mambrino',
  'description': 'British Thoroughbred racehorse'},
 'Beltenebros': {'id': 'Q4366360',
  'label': 'Prince of Shadows',
  'description': '1991 film'},
 'Dorotea': {'id': 'Q132334',
  'label': 'Dorotea Municipality',
  'description': 'municipality in Västerbotten County, Sweden'},
 'Fee': {'id': 'Q2627241',
  'label': 'Antoine Laurent Apollinaire Fée',
  'description': 'French botanist (1789-1874)'},
 'conde de Lemos': {'id': 'Q8349573',
  'label': 'Count of Lemos',
  'description': 'nobl

Après avoir filtré la liste avec WikiData, on s'aperçoit qu'il y a des lieux qui ont été détecté comme personnages par spaCy comme Limpias, des animaux aussi ont été détéctés comme personnages. Les résultats étant retourné en anglais moderne, on utilisera spaCy en anglais pour ne garder uniquement que les personnages.

In [19]:
# Chargement du modèle anglais pour traiter les descriptions Wikidata
nlp_en = spacy.load("en_core_web_md")

# Ensembles de mots-clés sémantiques (lemmas)
mots_valides = {"character", "novelist", "writer", "poet", "philosopher", 
                "hero", "warrior", "mythology", "mythical", "king", "author", "person","name"}

mots_exclusions = {"racehorse", "cycling", "municipality", "satellite", "genus", "stop", "town", "city", "film"}

def est_une_entite_valide(description):
    doc = nlp_en(description.lower())
    lemmas = {token.lemma_ for token in doc}
    
    # 1. On exclut si un mot banni est présent
    if lemmas.intersection(mots_exclusions):
        return False
    
    # 2. On valide si un mot clé de personnage/humain est présent
    if lemmas.intersection(mots_valides):
        return True
        
    return False

# Filtrage de ton dictionnaire de résultats
resultats_finaux_intro = {}

for nom_brut, info in resultats_validation.items():
    if est_une_entite_valide(info['description']):
        resultats_finaux_intro[nom_brut] = info

# Affichage du nombre de survivants
print(f"Entités après nettoyage sémantique : {len(resultats_finaux_intro)} / {len(resultats_validation)}")

Entités après nettoyage sémantique : 39 / 76


In [20]:
resultats_finaux_intro

{'Miguel de Cervantes Saavedra': {'id': 'Q5682',
  'label': 'Miguel de Cervantes',
  'description': 'Spanish novelist, poet, and playwright (1547-1616)'},
 'Marcela': {'id': 'Q19798827',
  'label': 'Marcela',
  'description': 'female given name'},
 'Sancho Panza': {'id': 'Q630823',
  'label': 'Sancho Panza',
  'description': 'fictional character in the novel Don Quixote'},
 'Sansón Carrasco': {'id': 'Q6120178',
  'label': 'Sansón Carrasco',
  'description': 'fictional character from the novel Don Quixote'},
 'Dulcinea del Toboso': {'id': 'Q1264776',
  'label': 'Dulcinea',
  'description': 'fictional character from Don Quixote'},
 'Dulcinea': {'id': 'Q911015',
  'label': 'Dulcinea',
  'description': 'female given name'},
 'Camacho': {'id': 'Q21482925',
  'label': 'Camacho',
  'description': 'family name'},
 'Basilio': {'id': 'Q18416923',
  'label': 'Basilio',
  'description': 'male given name'},
 'Pedro': {'id': 'Q15897419',
  'label': 'Pedro',
  'description': 'male given name'},
 'Ben

#### **Détection des personnages présents dans le texte**

Après avoir pu établir une liste des personnages présents dans la partie introductive de Don Quixote, il faut maintenant voir est-ce que spaCy arrive a détecter la liste des personnages présents dans le texte de Don Quixote :

In [21]:
''' 
Reprise du dernier code utilisé pour détécté la liste des personnages présents dans la partie introductive
en modifiant les paramètres pour cette fois récupérer la liste de personnages présents dans le texte.
'''

# 1. Correction de l'extraction : on boucle sur chaque doc de la liste
chapitres_roman = quijote_docs[2::2]

entites_brutes_corps_texte = [
    ent.text.replace('\n', ' ').strip() 
    for doc in chapitres_roman 
    for ent in doc.ents 
    if ent.label_ == "PER"
]

# 2. Nettoyage initial (inchangé)
bruit_ner = {"Del", "Donde", "Que", "Al", "Prólogo", "Capítulo", "Folio"}

entites_nettoyees_texte = []
for nom in entites_brutes_corps_texte:
    nom_propre = re.sub(r'[?.,;!:]+$', '', nom)
    if nom_propre not in bruit_ner and len(nom_propre) > 2:
        entites_nettoyees_texte.append(nom_propre)

# 3. Réduction par inclusion (inchangé)
noms_uniques_tries_corps_texte = sorted(list(set(entites_nettoyees_texte)), key=len, reverse=True)
mapping_normalisation_texte = {}

for i, nom_long in enumerate(noms_uniques_tries_corps_texte):
    for nom_court in noms_uniques_tries_corps_texte[i+1:]:
        if nom_court in nom_long:
            mapping_normalisation_texte[nom_court] = nom_long

# 4. Application du mapping et comptage
texte_normalise = [mapping_normalisation_texte.get(n, n) for n in entites_nettoyees_texte]
stats_texte = Counter(texte_normalise)

# 5. Affichage
print(f"Nombre d'entités uniques après réduction : {len(stats_texte)}")
print("-" * 50)
for personnage, freq in stats_texte.most_common(15):
    print(f"{personnage:<35} | {freq}")
    

Nombre d'entités uniques après réduction : 1481
--------------------------------------------------
Sanchos                             | 1751
Amigo Sancho Panza                  | 268
Dulcineas                           | 169
Diose                               | 146
Rióse Camila                        | 136
Notó Anselmo                        | 131
el Lotario                          | 127
Reparó Dorotea                      | 98
Dulcinea del Toboso                 | 97
Mas don Fernando                    | 83
señora Luscinda                     | 81
lela Zoraida                        | 67
Señoras                             | 63
Mas Pedro                           | 57
''Cardenio                          | 56


SpaCy a détecté en tout 1482 personnages mais en observant le top 15 des noms les plus données, il y a des verbes qui sont détectés comme personnages (Diose, Riose). Cela n'est pas surprenant étant donné que le texte est rédigé en espagnol ancien et que ces verbes ont leurs premières lettres en majuscule : ils se situaient donc en début de phrases. On note aussi que spaCy même si spaCy détecte correctement des personnages, il colle les préfixe culturel au nom. Par exemple dans lela Zoraida, "lela" est un titre respectueux utilisé par les personnes maures dans le roman. Pour essayer de réduire au maximum cette liste de 1482 personnages sans perdre en qualité, cette liste sera réduite au personnages étant mentionnés plus de 5 fois :

In [22]:
# 1. On définit les nouveaux bruits détectés dans le roman
prefixes_roman = [
    "Amigo", "Mas", "Rióse", "Notó", "Reparó", "Dijo", "el", "la", "lela", "señora"
]

def nettoyage_roman(nom_brut):
    # Nettoyage de base
    n = nom_brut.replace('\n', ' ').strip()
    n = re.sub(r'[?.,;!:]+$', '', n)
    
    # Retrait des nouveaux préfixes
    for p in prefixes_roman:
        n = re.sub(f"^{p}\\s+", "", n, flags=re.IGNORECASE)
        
    # Correction des pluriels fréquents (Sanchos -> Sancho)
    if n.endswith('s') and n[:-1] in ["Sancho", "Dulcinea"]:
        n = n[:-1]
        
    return n.strip()

# 2. Application du nettoyage et filtrage par fréquence
entites_textes_nettoyees = [nettoyage_roman(n) for n in entites_brutes_corps_texte]
stats_brutes_texte = Counter(entites_textes_nettoyees)

# 3. On ne garde que les personnages avec au moins 5 occurrences pour Wikidata
noms_pertinents = {nom: count for nom, count in stats_brutes_texte.items() if count >= 5}

print(f"Nombre d'entités après filtrage (freq >= 5) : {len(noms_pertinents)}")
print("-" * 50)
for nom, count in Counter(noms_pertinents).most_common(10):
    print(f"{nom:<30} | {count}")

Nombre d'entités après filtrage (freq >= 5) : 151
--------------------------------------------------
Sancho                         | 1751
Sancho Panza                   | 268
Dulcinea                       | 173
Dios                           | 146
Camila                         | 136
Anselmo                        | 130
Lotario                        | 128
Dorotea                        | 98
Dulcinea del Toboso            | 86
don Fernando                   | 83


In [23]:
print(len(noms_pertinents),noms_pertinents, sep="/n")

151/n{'Apolo': 6, 'Rocinante': 13, 'señor caballero': 25, 'Preguntáronle': 6, 'prosupuesto': 7, 'Dulcinea': 173, 'Viendo': 25, 'Díjole': 5, 'Dios': 146, 'Señor caballero': 8, 'Él': 26, 'Andrés': 18, 'Venid': 8, 'Dulcinea del Toboso': 86, 'Valdovinos': 5, 'Rodrigo de Narváez': 5, 'Satanás': 10, 'Barrabás': 5, 'Nicolás': 14, 'Amadís de Gaula': 12, 'Amadís': 20, 'Hízolo': 18, 'Alejandro': 6, 'Homero': 6, 'recebí': 6, 'Hiciéronlo': 5, 'Sancho Panza': 268, 'Sancho': 1751, 'Panza': 17, 'Calla': 14, 'Mire': 8, 'Preguntéle': 5, 'Está': 7, 'Cide Hamete Benengeli': 9, 'Paréceme': 12, 'osaré': 7, 'Mambrino': 15, 'Ansí': 10, 'Antonio': 42, 'Grisóstomo': 27, 'Marcela': 23, 'Ambrosio': 13, 'Pedro': 55, 'Vivaldo': 10, 'Finalmente': 24, 'Señor': 62, 'Roldán': 11, 'Dígolo': 12, 'Querría': 6, 'Sancho amigo': 6, 'Púsose': 5, 'Diole': 6, 'Nuestro Señor': 10, 'Parecióle': 6, 'Sucedió': 5, 'Dígote': 5, 'arzón': 7, 'quisiéredes': 10, 'Mandó': 7, 'Elena': 5, 'Llegó': 6, 'Fernando': 33, 'Luscinda': 81, 'Carden

Cette liste a été réduite de près de 90% mais ont remarque quand même des doublons dans le TOP 10 (Sancho & Sancho Panza ou Dulcinea et Duclinea) et toujours beaucoup de verbes. Maintenant vérifions combien sur les 151 personnages détectés par spaCy sont répertoriés sur Wikipédia :

In [24]:
import requests
import time

def valider_entite_wikidata(nom_entite):
    url = "https://www.wikidata.org/w/api.php"
    headers = {
        'User-Agent': 'ProjetEtudeMasterLorame/1.0 (contact: lorame@exemple.com)'
    }
    params = {
        "action": "wbsearchentities",
        "language": "es",
        "format": "json",
        "search": nom_entite,
        "type": "item"
    }
    
    try:
        reponse = requests.get(url, params=params, headers=headers, timeout=5)
        if reponse.status_code == 200:
            donnees = reponse.json()
            if donnees.get("search"):
                match = donnees["search"][0]
                return {
                    "id": match["id"],
                    "label": match["label"],
                    "description": match.get("description", "No hay descripción disponible")
                }
    except Exception as e:
        print(f"Erreur de connexion pour {nom_entite} : {e}")
    return None

# Utilisation de la liste des 151 noms pertinents du corps du texte
validation_corps_texte = {}

print(f"Début de la validation Wikidata pour les {len(noms_pertinents)} entités du roman...")

# On itère sur noms_pertinents (ton dictionnaire de 151 noms)
for i, nom in enumerate(noms_pertinents.keys()):
    # Pause pour respecter les limites de l'API
    time.sleep(0.1) 
    
    info_wiki = valider_entite_wikidata(nom)
    
    if info_wiki:
        validation_corps_texte[nom] = info_wiki
        print(f"[{i+1}/{len(noms_pertinents)}] OK : {nom} -> {info_wiki['label']} -> {info_wiki['description']}")
    else:
        print(f"[{i+1}/{len(noms_pertinents)}] KO : {nom}")

print(f"\nValidation terminée. {len(validation_corps_texte)} entités trouvées sur {len(noms_pertinents)}.")

Début de la validation Wikidata pour les 151 entités du roman...
[1/151] OK : Apolo -> Apolo -> male given name
[2/151] OK : Rocinante -> Rocinante -> horse of Don Quixote
[3/151] KO : señor caballero
[4/151] KO : Preguntáronle
[5/151] KO : prosupuesto
[6/151] OK : Dulcinea -> Dulcinea -> female given name
[7/151] OK : Viendo -> Viendorf -> locality in Hollabrunn District
[8/151] OK : Díjole -> Dijolé D. Quijote: sancho amigo, la noche se nos va entrando... -> print in the National Gallery of Art (NGA 132298)
[9/151] OK : Dios -> deity -> natural or supernatural god or goddess, divine being
[10/151] KO : Señor caballero
[11/151] OK : Él -> Greece -> country in Southeast Europe
[12/151] OK : Andrés -> Andrés -> male given name
[13/151] OK : Venid -> Venidium -> genus of plants
[14/151] OK : Dulcinea del Toboso -> Dulcinea -> fictional character from Don Quixote
[15/151] OK : Valdovinos -> Valdovinos -> family name
[16/151] KO : Rodrigo de Narváez
[17/151] OK : Satanás -> Satan -> entity

Wikidata confirme que la liste de personnages détectés est cohérente et fait intéressant même quand le nom entier n'est pas présenté Wikidata reconnait le bon personnage (exemple : chez spaCy : Pedro Recio, Wikidata reconnait Pedro Recio Aguero, Dulcinea et Dulcinea del Toboso), il faut encore filtrer d'avantage la liste en utilisant les id de personnages sur Wikidata ainsi peut importe l'appelation d'un personnage si c'est le même id ça sera compté comme un personnage unique. On note aussi que Wikidata force la correspondance avec notemment : Él -> Greece -> country in Southeast Europe ou OK : Viendo -> Viendorf -> locality in Hollabrunn District. Il faudra encore filtrer la liste d'avantage. Pour ce faire, on va utilisé la Zero-Shot classification pour prendre la liste de personnages définitif en fonction des description fournies par Wikidata.

Pour utiliser la Zero-Shot Classification, il faudra télécharger la bibliothèque Transformers de Hugging Face et on utilisera le modèle facebook/bart-large-mnli qui est perfomant pour la classification de texte en anglais :

In [41]:
from transformers import pipeline

# Chargement d'un modèle qui comprend réellement l'espagnol
# Il est un peu plus lourd mais bien plus précis pour Cervantes
zero_shot_multilingue = pipeline(
    "zero-shot-classification", 
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

def classification_robuste(description):
    if not description or len(description) < 5:
        return "uncertain"
    
    # On simplifie les labels au maximum
    labels_simples = ["persona", "lugar", "cosa"]
    
    res = zero_shot_multilingue(
        description, 
        labels_simples, 
        hypothesis_template="Este texto trata de una {}."
    )
    
    # On récupère le meilleur
    label_top = res['labels'][0]
    score_top = res['scores'][0]
    
    # Seuil de secours à 0.30 pour ne rien rater
    if score_top > 0.30:
        return label_top
    return "uncertain"

# Application
personnages_identifies = []
for nom, info in validation_corps_texte.items():
    cat = classification_robuste(info['description'])
    if cat == "persona":
        personnages_identifies.append({
            "nom": nom,
            "desc": info['description'],
            "wiki_id": info['id']
        })

print(f"Nombre de personnages validés : {len(personnages_identifies)}")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 27950.96it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Nombre de personnages validés : 7


Avec cette méthode, seulement 7 personnages de la liste sont validés par Hugging Face. 

## **Résumé**

Après avoir listé les personnages, on tente de faire des résumés de chapitres :

In [49]:
from collections import Counter
from heapq import nlargest

# 1. On cible le segment souhaité
segment_quichotte = quijote_docs[2]

# 2. Extraction et filtrage des lemmes (on ignore le "bruit" linguistique)
# On utilise .lemma_ car en espagnol, c'est crucial pour regrouper les concepts
mots_frequence = [
    token.lemma_.lower() for token in segment_quichotte 
    if not token.is_stop and not token.is_punct and not token.is_space
]

# 3. Calcul de l'importance statistique
compteur_mots = Counter(mots_frequence)
if compteur_mots:
    poids_max = max(compteur_mots.values())
    # Normalisation des scores
    scores_mots = {mot: freq / poids_max for mot, freq in compteur_mots.items()}
else:
    scores_mots = {}

# 4. Scoring des phrases par sommation des poids
# Mathématiquement : $$Score(S) = \sum_{w \in S} \text{Poids}(w)$$
scores_phrases = {}
for phrase in segment_quichotte.sents:
    for mot in phrase:
        lemme = mot.lemma_.lower()
        if lemme in scores_mots:
            if phrase not in scores_phrases:
                scores_phrases[phrase] = scores_mots[lemme]
            else:
                scores_phrases[phrase] += scores_mots[lemme]

# 5. Extraction du Top 5
meilleures_phrases = nlargest(5, scores_phrases, key=scores_phrases.get)

# Tri chronologique pour garder la cohérence du récit
resume_ordonne = sorted(meilleures_phrases, key=lambda p: p.start)

print(f"RÉSUMÉ DU SEGMENT INDEX 2 (5 PHRASES)\n")
for i, ph in enumerate(resume_ordonne, 1):
    print(f"{i}. {ph.text.strip()}\n")

RÉSUMÉ DU SEGMENT INDEX 2 (5 PHRASES)

1. Autores hay que dicen que la
primera aventura que le avino fue la del Puerto Lápice; otros dicen que la
de los molinos de viento; pero, lo que yo he podido averiguar en este caso,
y lo que he hallado escrito en los Anales de la Mancha, es que él anduvo
todo aquel día, y, al anochecer, su rocín y él se hallaron cansados y
muertos de hambre; y que, mirando a todas partes por ver si descubriría
algún castillo o alguna majada de pastores donde recogerse y adonde pudiese
remediar su mucha hambre y necesidad, vio, no lejos del camino por donde
iba, una venta, que fue como si viera una estrella que, no a los portales,
sino a los alcázares de su redención le encaminaba.

2. Estaban acaso a la puerta dos mujeres mozas, destas que llaman del partido,
las cuales iban a Sevilla con unos arrieros que en la venta aquella noche
acertaron a hacer jornada; y, como a nuestro aventurero todo cuanto
pensaba, veía o imaginaba le parecía ser hecho y pasar al modo de

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# On récupère les chapitres 2 et 3 (indices 2 et 3 selon les tests précédents)
indices_chapitres = [2, 3]
corpus_quichotte_lemmes = []

for index in indices_chapitres:
    doc = quijote_docs[index]
    # On extrait les lemmes des mots qui ne sont ni des stop words ni de la ponctuation
    lemmes_nettoyes = [
        token.lemma_.lower() for token in doc 
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    corpus_quichotte_lemmes.append(" ".join(lemmes_nettoyes))

In [30]:
# Initialisation du vectoriseur
vectoriseur_quichotte = TfidfVectorizer()

# Apprentissage et transformation
matrice_sparse_quichotte = vectoriseur_quichotte.fit_transform(corpus_quichotte_lemmes)

# Extraction des noms de colonnes (le vocabulaire)
noms_lemmes = vectoriseur_quichotte.get_feature_names_out()

# Création de la table de résultats (indexée par chapitre)
table_tfidf_quichotte = pd.DataFrame(
    matrice_sparse_quichotte.toarray(),
    index=["Chapitre 2", "Chapitre 3"],
    columns=noms_lemmes
)

In [51]:
print("Top 10 - Chapitre 2 (La sortie) :")
print(table_tfidf_quichotte.loc["Chapitre 2"].sort_values(ascending=False).head(10))

print("\n" + "="*30 + "\n")

print("Top 10 - Chapitre 3 (L'adoubement) :")
print(table_tfidf_quichotte.loc["Chapitre 3"].sort_values(ascending=False).head(10))

Top 10 - Chapitre 2 (La sortie) :
él           0.436990
quijote      0.218495
ver          0.218495
don          0.218495
venta        0.201688
caballero    0.168073
poder        0.168073
castillo     0.151266
arma         0.134459
parecer      0.117651
Name: Chapitre 2, dtype: float64


Top 10 - Chapitre 3 (L'adoubement) :
capítulo    0.707107
iii         0.707107
paje        0.000000
pan         0.000000
papelón     0.000000
para        0.000000
parecer     0.000000
parecido    0.000000
parte       0.000000
partido     0.000000
Name: Chapitre 3, dtype: float64


Phrase 1 (Chapitre 2) : Les coefficients du Chapitre 2 traduisent mathématiquement la distorsion de réalité du protagoniste, où des termes comme "venta" (0.20) et "castillo" (0.15) émergent avec des poids similaires, prouvant que l'auberge et la forteresse occupent une importance sémantique équivalente dans ce segment du corpus.

Phrase 2 (Chapitre 3) : À l'inverse, les vecteurs du Chapitre 3 révèlent un biais de sparsité extrême où les métadonnées structurales "capítulo" et "iii" saturent la variance (0.71), indiquant que le bruit organisationnel occulte totalement la richesse lexicale de la cérémonie d'adoubement dans cette extraction.

## **Traduction**

Après avoir implémenté quelques matrices TD-IDF, on va utiliser un modèle de traduction automatique, dit NMT local via Hugging Face pour traduire quelques chapitres du texte.

In [ ]:
#pip install "transformers[sentencepiece]"

Note: you may need to restart the kernel to use updated packages.


In [33]:
from transformers import MarianMTModel, MarianTokenizer

# 1. Chargement explicite du modèle et du tokenizer
nom_modele_traduction = "Helsinki-NLP/opus-mt-es-en"
quichotte_tokenizer = MarianTokenizer.from_pretrained(nom_modele_traduction)
quichotte_modele_marian = MarianMTModel.from_pretrained(nom_modele_traduction)

def traduire_segment_espagnol(texte_es):
    # Encodage du texte espagnol
    inputs_encodes = quichotte_tokenizer(texte_es, return_tensors="pt", padding=True)
    
    # Génération de la traduction
    outputs_traduits = quichotte_modele_marian.generate(**inputs_encodes)
    
    # Décodage du résultat vers l'anglais
    texte_anglais = quichotte_tokenizer.batch_decode(outputs_traduits, skip_special_tokens=True)
    return texte_anglais[0]

# Test rapide
print(traduire_segment_espagnol("En un lugar de la Mancha"))

/Users/loram/Documents/NLP_AOW_Projet/nlp_aow/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 69189.93it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In a place in La Mancha


In [34]:
# On utilise le modèle MarianMT que tu viens de charger avec succès
chapitre_espagnol = quijote_docs[2]
phrases_traduites = []

print(f"Début de la traduction du Chapitre 2 ({len(list(chapitre_espagnol.sents))} phrases)...")

for i, phrase in enumerate(chapitre_espagnol.sents):
    texte_brut = phrase.text.strip()
    
    # On évite les segments trop courts ou vides
    if len(texte_brut) > 2:
        # Encodage et génération
        entrees = quichotte_tokenizer(texte_brut, return_tensors="pt", padding=True)
        sorties = quichotte_modele_marian.generate(**entrees)
        traduction = quichotte_tokenizer.batch_decode(sorties, skip_special_tokens=True)[0]
        
        phrases_traduites.append(traduction)
    
    # Indicateur de progression tous les 10 segments
    if i % 10 == 0:
        print(f"Progression : {i} phrases traitées.")

# Reconstruction du chapitre en anglais
quijote_ch2_en = " ".join(phrases_traduites)

print("\n--- Traduction terminée ---")
print(f"Aperçu final : {quijote_ch2_en[:300]}...")

Début de la traduction du Chapitre 2 (38 phrases)...
Progression : 0 phrases traitées.
Progression : 10 phrases traitées.
Progression : 20 phrases traitées.
Progression : 30 phrases traitées.

--- Traduction terminée ---
Aperçu final : That it is about the first way out of his land that the ingenious Don Quixote Hechas made, because, these preventions, he did not want to wait any longer to put in effect his thought, pressing to it the lack that he thought he was making his delay in the world, according to the grievances that he in...


In [35]:
quijote_ch2_en

'That it is about the first way out of his land that the ingenious Don Quixote Hechas made, because, these preventions, he did not want to wait any longer to put in effect his thought, pressing to it the lack that he thought he was making his delay in the world, according to the grievances that he intended to undo, one-eyed ones to straighten out, without reasons to emend, and abuses to improve and debts to satisfy. And so, without giving any part to any person of his intention, and without anyone seeing him, one morning, before the day, which was one of the hot ones of the month of July, he armed himself with all his weapons, went up upon Rocinante, put on his ill made-up cell, clutched his plow, took his spear, and, through the false door of a pen, went out into the field with great joy and exultation to see how easily he had begun his good desire. But he hardly saw himself in the field, when he was struck by a terrible thought, and such, that he would almost make him leave the begun

Un peu d'analyse de sentiments sur les textes traduits :

In [36]:
from transformers import pipeline
analyseur_sentiment_en = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Test sur les 5 premières phrases traduites
for phrase in phrases_traduites[:5]:
    print(f"Phrase : {phrase[:50]}... | Sentiment : {analyseur_sentiment_en(phrase)[0]}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9245.41it/s]


Phrase : That it is about the first way out of his land tha... | Sentiment : {'label': 'NEGATIVE', 'score': 0.9821236729621887}
Phrase : And so, without giving any part to any person of h... | Sentiment : {'label': 'POSITIVE', 'score': 0.9992831349372864}
Phrase : But he hardly saw himself in the field, when he wa... | Sentiment : {'label': 'NEGATIVE', 'score': 0.9933320879936218}
Phrase : These thoughts caused him to waver in his purpose;... | Sentiment : {'label': 'POSITIVE', 'score': 0.9928176403045654}
Phrase : With regard to the white weapons, he planned to cl... | Sentiment : {'label': 'POSITIVE', 'score': 0.9524713158607483}


In [37]:
# On récupère les 20 derniers segments pour être sûr d'avoir 10 chapitres de contenu
segments_finaux = quijote_docs[-20:]

# On ne garde que les segments de contenu (les indices impairs dans cette sélection)
# Si ton dernier segment [-1] est du contenu, alors [-1], [-3], [-5]... le sont aussi.
contenus_seuls = segments_finaux[1::2] 

print(f"Nombre de chapitres de contenu récupérés : {len(contenus_seuls)}")

from collections import Counter
from heapq import nlargest

def resume_et_traduit(doc_spacy, modele, tokenizer, nb_phrases=2):
    # --- ÉTAPE 1 : RÉSUMÉ EXTRACTIF ---
    mots = [t.lemma_.lower() for t in doc_spacy if not t.is_stop and not t.is_punct and not t.is_space]
    if not mots: return ""
    
    freq = Counter(mots)
    f_max = max(freq.values())
    scores_mots = {m: c/f_max for m, c in freq.items()}
    
    scores_phrases = {}
    for sent in doc_spacy.sents:
        for token in sent:
            if token.lemma_.lower() in scores_mots:
                scores_phrases[sent] = scores_phrases.get(sent, 0) + scores_mots[token.lemma_.lower()]
    
    meilleures_phrases = nlargest(nb_phrases, scores_phrases, key=scores_phrases.get)
    resume_es = " ".join([p.text.strip() for p in sorted(meilleures_phrases, key=lambda p: p.start)])
    
    # --- ÉTAPE 2 : TRADUCTION NEURONALE ---
    # On découpe en phrases pour le modèle de traduction
    doc_resume = nlp_es(resume_es)
    phrases_en = []
    
    for sent in doc_resume.sents:
        # Encodage et génération avec le modèle MarianMT
        entrees = tokenizer(sent.text, return_tensors="pt", padding=True, truncation=True)
        sorties = modele.generate(**entrees)
        traduction = tokenizer.batch_decode(sorties, skip_special_tokens=True)[0]
        phrases_en.append(traduction)
        
    return " ".join(phrases_en)

# Application sur les 10 chapitres de contenu
resultats_finaux_en = []

for i, doc in enumerate(contenus_seuls):
    num_chapitre = 74 - (len(contenus_seuls) - 1 - i)
    print(f"Traitement du Chapitre {num_chapitre}...")
    
    traduction_finale = resume_et_traduit(doc, quichotte_modele_marian, quichotte_tokenizer)
    resultats_finaux_en.append({
        "chapitre": num_chapitre,
        "resume_en": traduction_finale
    })

Nombre de chapitres de contenu récupérés : 10
Traitement du Chapitre 65...
Traitement du Chapitre 66...
Traitement du Chapitre 67...
Traitement du Chapitre 68...
Traitement du Chapitre 69...
Traitement du Chapitre 70...
Traitement du Chapitre 71...
Traitement du Chapitre 72...
Traitement du Chapitre 73...
Traitement du Chapitre 74...


In [57]:
# On parcourt la liste des dictionnaires pour afficher chaque chapitre

print("SYNTHÈSE FINALE - DON QUIXOTE (ENGLISH VERSION)")


for item in resultats_finaux_en:
    print(f"CHAPITRE {item['chapitre']} :")
    print(f"Summary: {item['resume_en']}")
    print("/n")

SYNTHÈSE FINALE - DON QUIXOTE (ENGLISH VERSION)
CHAPITRE 65 :
Summary: Know, sir, that I am called the bachelor Samson Carrasco; I am of the same place as Don Quixote of La Mancha, whose folly and folly moves us to pity all who know him, and among those who have had it the most; and, believing that his health is in his rest and that he is in his land and in his house, I gave a trace to make him stay in it; and so, there will be three months that I went out on the way as a knight-errant, calling me the Knight of the Mirrors, with the intention of fighting with him and beating him, without harming him, putting as a condition of our fight that the vanquished man should remain at the discretion of the victor; and what I intended to ask him, because I already judged him to be defeated, was that he should return to his place and that he should not give him out in a whole year, in which time he could be healed; but the fate ordered him otherwise, because he overcame me and knocked me off my h

In [59]:

tableau_resumes_quichotte = pd.DataFrame(resultats_finaux_en)

#
tableau_resumes_quichotte.to_csv('resumes_quichotte_en.csv', index=False, encoding='utf-8')

print("Le fichier 'resumes_quichotte_en.csv' a été généré dans le dossier data.")

Le fichier 'resumes_quichotte_en.csv' a été généré dans le dossier data.
